# 9 - Preparing the deposits

**What it establishes.** Produces the three artefacts published with the paper, starting from the
processed objects the `0*` notebooks leave behind. It analyses nothing: it centralises, slims,
verifies and packages.

**Inputs.** The eleven `*_processed*.h5ad` objects of the five datasets.

**Outputs.**

| Bundle | Contents | Destination |
|---|---|---|
| `deposits/zenodo_data/` | the eleven slimmed objects, ~32.9 GB upper bound | Zenodo (data record) |
| `deposits/zenodo_code/` | notebooks with outputs, final tables and figures, container image | Zenodo (code record) |
| `deposits/cellxgene/` | the in-house object adapted to schema 7.1.0 | CELLxGENE Discover |

**Prerequisites.** The project environment installed (`pip install -e .`) and the processed objects
on disk. Nothing here needs a GPU or the FASTQ files.

**Re-entry point it defines.** The data bundle is what lets a third party run from notebook `1AA`
onwards without downloading a single FASTQ. Everything before that - STARsolo, CellBender,
normalisation - is only reproducible from `workflow/`.

The logic lives in `kranocyte.submissions`; this
notebook only orchestrates (`CODING_GUIDELINES.md` §22, `CODE-NB-001`).

## Setup

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

import anndata as ad
import pandas as pd

# (see docs/MIGRATION_PLAN.md). Change both lines then.
from kranocyte.config import ROOT, DATA_PROCESSED, FIGURES, processed_sources
from kranocyte.submissions import (
    SPEC_DEFAULT,
    CXG_SCHEMA_VERSION,
    CXG_MOUSE_ANNOTATION,
    slim_all,
    write_manifest,
    verify_manifest,
    cxg_report,
    gencode_symbol_map,
    cxg_build,
    sha256_of,
)

print(f"project root: {ROOT}")

## Parameters

Everything that can be changed is here (`CODE-NB-007`). The rest of the notebook uses these; it
does not redefine them further down.

In [ ]:
# --- editable parameters ----------------------------------------------------
DEPOSITS   = ROOT / "deposits"           # root of the three bundles
OVERWRITE  = False                       # True to rebuild a bundle already written
DRY_RUN    = True                        # True: report only, do not write the h5ad files

# The object going to CELLxGENE: the full in-house dataset.
CXG_SOURCE = "AA_inhouse_dataset_processed.h5ad"

# Source accessions the derived deposit MUST cite.
SOURCE_ACCESSIONS = {
    "oprescu_2020":     "GSE138826",
    "song_2023":        "GSE215922",
    "southerland_2023": "GSE227075",
    "nicoletti_2023":   "GSE221736",
    "inhouse":          "ENA accession pending (see submission/ena/)",
}
# ---------------------------------------------------------------------------

ZENODO_DATA = DEPOSITS / "zenodo_data"
ZENODO_CODE = DEPOSITS / "zenodo_code"
CELLXGENE   = DEPOSITS / "cellxgene"

## Inputs and checks

Before touching anything, locate the eleven objects and confirm they are where expected.

In [ ]:
# processed_sources() finds the objects in both the current layout
# (data/ARAUZO_03/processed_adatas and data/public_scRNAseq/datasets/*/processed_adatas)
# and the migration target layout (data/processed/<dataset>).
sources = processed_sources()
inventory = pd.DataFrame(
    [{"file": p.name, "dataset": p.parent.parent.name, "GB": p.stat().st_size / 1e9}
     for p in sources]
).sort_values("GB", ascending=False)

assert len(sources) == 11, f"expected 11 objects, found {len(sources)}"
print(f"total at source: {inventory.GB.sum():.2f} GB")
inventory

## Slimming

`SPEC_DEFAULT` removes three layers from every object: `norm_scran_log1p`, `norm_log1p` and
`norm_cpm_log1p`. Nothing is lost - the first **is `X`** (they match row by row once indices are
sorted) and the other two are produced and never consumed anywhere.

`counts_raw` and the velocity layers are **kept in all eleven objects**, even though they are not
used downstream of notebook 1: there is ample headroom in the record, so anyone can audit what
CellBender removed or run RNA velocity on a dataset for which the project did not.

From 48.65 to **32.88 GB as an upper bound**, leaving 17.12 GB of headroom against Zenodo's 50 GB
limit. The bound does not credit recompression: the verified object came out at 19.5 %, not the
67 % the bound predicted. Detail and rationale in `docs/H5AD_SLIMMING_SPEC.md`.

On the deposited objects, `prepare_cpdb_input()` detects that `norm_scran_log1p` is absent, sees
that `uns['normalization']` records scran, and uses `expm1(X)`, which is the same matrix.

In [ ]:
for src in sources:
    drop = SPEC_DEFAULT.layers_to_drop(src.name, ad.read_h5ad(src, backed="r").layers.keys())
    print(f"{src.name:42s} drops {len(drop)} layers: {sorted(drop)}")

In [ ]:
if DRY_RUN:
    print("DRY_RUN=True: nothing was written. Set it to False to build the bundle.")
    records = []
else:
    records = slim_all(sources, ZENODO_DATA, spec=SPEC_DEFAULT, overwrite=OVERWRITE)
    pd.DataFrame(records)

## Manifests

Each bundle carries its manifest with path, size and sha256. It is what lets a download be checked
against what was published, and the manifest is tracked in git even though the data is not.

In [ ]:
if not DRY_RUN:
    write_manifest(
        ZENODO_DATA,
        extra={
            "project": "kranocyte",
            "commit": __import__("subprocess").run(
                ["git", "rev-parse", "--short", "HEAD"], cwd=ROOT,
                capture_output=True, text=True).stdout.strip(),
            "slimming_spec": "docs/H5AD_SLIMMING_SPEC.md",
            "source_accessions": "; ".join(f"{k}={v}" for k, v in SOURCE_ACCESSIONS.items()),
            "re_entry_point": "notebook 1AA onwards",
            "cpdb_note": "prepare_cpdb_input falls back to expm1(X); X is the scran+log1p matrix",
        },
    )
    verify_manifest(ZENODO_DATA)

## Code and results bundle

Notebooks **with** their outputs (in git they go without, `CODE-NB-004`), the final tables, the
paper figures and the container image that produced the matrices.

The `.sif` is included because a registry tag can be repointed or deleted, which is exactly the
failure the deposit protects against. Before including it, its redistribution terms must be
resolved: its `org.opencontainers.image.licenses` label is empty (`CODE-LIC-005`).

In [ ]:
CODE_BUNDLE = {
    "notebooks_with_outputs": ROOT / "notebooks" / "rendered",
    "final_tables":          ROOT / "results" / "tables",
    "paper_figures":         FIGURES / "paper",
    "container":             ROOT / "data" / "external" / "containers" / "star_2.7.11b.sif",
}
for label, path in CODE_BUNDLE.items():
    exists = path.exists()
    size = sum(f.stat().st_size for f in path.rglob("*") if f.is_file()) if path.is_dir() else (
        path.stat().st_size if exists else 0)
    print(f"{label:24s} {'OK ' if exists else 'MISSING'} {size/1e9:6.2f} GB  {path}")

## CELLxGENE Discover

Schema **7.1.0**, checked on 2026-11-20.

The real obstacle is not the metadata fields but the gene annotation: the schema **pins**
`GENCODE vM37 (GRCm39), Ensembl 114` for *Mus musculus*, and this project quantified against **NCBI
RefSeq GRCm39**. Mapping RefSeq symbols to Ensembl identifiers is not a rename: it loses genes and
needs review. `kranocyte.io.add_ensembl_ids` and `download_gene2ensembl` are the starting point, but
the target has to be GENCODE vM37 specifically.

`cxg_report()` only reports: it does not modify the object.

In [ ]:
adata_cxg = ad.read_h5ad(next(p for p in sources if p.name == CXG_SOURCE), backed="r")
report = cxg_report(adata_cxg)

In [ ]:
# Fields still pending a decision, and what is missing in each.
pending = pd.DataFrame([
    ("tissue_ontology_term_id", "UBERON term for the specific muscle; which one was dissected is still to be confirmed"),
    ("development_stage_ontology_term_id", "mouse developmental stage; pending the animal facility record"),
    ("disease_ontology_term_id", "PATO:0000461 (normal) if the samples are untreated controls"),
    ("cell_type_ontology_term_id", "map each obs['cell_type'] label to Cell Ontology"),
    ("donor_id", "animal identifier; AZ7845 and AZ7846 are libraries, not necessarily animals"),
    ("var index", "convert from NCBI RefSeq symbols to GENCODE vM37 identifiers"),
], columns=["field", "what is missing"])
pendientes

### Building the object

The mapping goes **through gene symbols**, because the object's `var['ensembl_id']` column is
empty: `add_ensembl_ids` was never filled in. Genes absent from GENCODE vM37 are dropped.

On the in-house dataset that means losing **4,732 of 25,575 genes (18.5 %)**, but the loss is
concentrated where it matters least: 4,107 are lncRNAs, 4,141 start with `Gm` (predicted genes) and
only **342 are protein-coding**. Their median detection is 24 cells against 535 among those kept,
and of the 2,000 variable genes triku selects, only 121 are lost.

In [ ]:
gene_map = gencode_symbol_map(ROOT / "data" / "external" / "annotation" / "gencode.vM37.gtf.gz")

# Schema fields the curator must annotate. The missing ones are listed above, in `pending`.
OBS_VALUES = {
    "organism_ontology_term_id": "NCBITaxon:10090",
    "sex_ontology_term_id": "PATO:0000384",          # male, from Xist and Y-chromosome genes
    "assay_ontology_term_id": "EFO:0009922",         # 10x 3' v3 (the SI-TT indices indicate v3.1)
    "self_reported_ethnicity_ontology_term_id": "na",
    "suspension_type": "cell",
    "is_primary_data": True,
    "tissue_type": "tissue",
    # pending: tissue_ontology_term_id, development_stage_ontology_term_id,
    #             disease_ontology_term_id, cell_type_ontology_term_id, donor_id
}

In [ ]:
if not DRY_RUN:
    adata_full = ad.read_h5ad(next(p for p in sources if p.name == CXG_SOURCE))
    cxg = cxg_build(
        adata_full,
        gene_map,
        out_path=CELLXGENE / "kranocyte_inhouse_cxg.h5ad",
        obs_values=OBS_VALUES,
        title="Kranocyte and FAP populations in mouse skeletal muscle (control)",
    )
    write_manifest(CELLXGENE, extra={"schema": CXG_SCHEMA_VERSION,
                                     "annotation": CXG_MOUSE_ANNOTATION})

## Summary

What is done and what remains to be decided before anything is submitted.

In [ ]:
summary = pd.DataFrame([
    ("zenodo_data", f"{inventory.GB.sum():.2f} GB -> ~32.9 GB (upper bound)", "ready; run with DRY_RUN=False"),
    ("zenodo_code", "notebooks, tables, figures, container", "pending: .sif licence"),
    ("cellxgene",   f"schema {CXG_SCHEMA_VERSION}", "pending: gene mapping and 6 obs fields"),
], columns=["bundle", "contents", "status"])
resumen